# Tesseract Experimentation Notebook — CNIE OCR

**Goal:** Run structured experiments to understand exactly what Tesseract can and cannot read on the CNIE card — before building any pipeline.

Every cell is an experiment. Run them one by one. Record what works and what doesn't.

---
## Setup — Install & Import

In [ ]:
# Run this ONCE to install dependencies
!pip install pytesseract opencv-python pillow pandas matplotlib

In [ ]:
# NOTHING TO RUN HERE — this is just a reminder:
#
# You need to install the Tesseract program on Windows BEFORE running the next cell.
# 1. Download from: https://github.com/UB-Mannheim/tesseract/wiki
# 2. Run the .exe installer
# 3. During install: check "Additional language data" → select Arabic + French
# 4. Default install path: C:\Program Files\Tesseract-OCR\tesseract.exe
#
# Once installed, move to the next cell.

In [ ]:
import pytesseract
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# SET YOUR TESSERACT PATH HERE (Windows)
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# Verify tesseract is working
try:
    version = pytesseract.get_tesseract_version()
    print(f"Tesseract version: {version}")
except Exception as e:
    print(f"ERROR: {e}")
    print("Make sure Tesseract is installed and the path above is correct")

In [ ]:
# Check which languages are installed
langs = pytesseract.get_languages()
print(f"Installed languages: {langs}")

needed = ['ara', 'fra', 'eng']
for lang in needed:
    status = 'OK' if lang in langs else 'MISSING — install it!'
    print(f"  {lang}: {status}")

---
## Section 1 — Load & Visualize the Card

Put your card images in the same folder as this notebook (`front.jpeg`, `back.jpeg`).

Your front image is rotated 90° — the code auto-fixes this.

In [ ]:
# UPDATE THESE PATHS to your actual card images
FRONT_PATH = "front.jpeg"
BACK_PATH  = "back.jpeg"   # change this to your actual back image filename

front = cv2.imread(FRONT_PATH)
back  = cv2.imread(BACK_PATH)

# Fix rotation — your phone photo is rotated sideways
if front is not None:
    front = cv2.rotate(front, cv2.ROTATE_90_COUNTERCLOCKWISE)
    print(f"Front loaded + rotated: {front.shape[1]}w x {front.shape[0]}h px")
else:
    print(f"ERROR: Could not load {FRONT_PATH}")
    print("Make sure the file exists in the same folder as this notebook")

if back is not None:
    back = cv2.rotate(back, cv2.ROTATE_90_COUNTERCLOCKWISE)
    print(f"Back loaded + rotated:  {back.shape[1]}w x {back.shape[0]}h px")
else:
    print(f"WARNING: Could not load {BACK_PATH}")
    print("Back side experiments will be skipped. That's OK for now.")

In [ ]:
# Display both sides
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

if front is not None:
    axes[0].imshow(cv2.cvtColor(front, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"Front — {front.shape[1]}x{front.shape[0]}")
else:
    axes[0].set_title("Front — NOT LOADED")

if back is not None:
    axes[1].imshow(cv2.cvtColor(back, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"Back — {back.shape[1]}x{back.shape[0]}")
else:
    axes[1].set_title("Back — NOT LOADED")

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

---
## Section 1.5 — Card Detection & Crop (Color-Based)

The CNIE card is GREEN. We detect it by its color using HSV color space.

**WARNING: When taking card photos, use a NON-GREEN background!**
- GOOD: white paper, dark desk, black surface, blue cloth
- BAD: green table, grass, green cloth

The green card must contrast with the background for detection to work.

In [ ]:
# Step 1: Visualize the card in HSV color space
# This helps us understand which green range to target

hsv = cv2.cvtColor(front, cv2.COLOR_BGR2HSV)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
axes[0].imshow(cv2.cvtColor(front, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")
axes[1].imshow(hsv[:,:,0], cmap='hsv')
axes[1].set_title("Hue (color)")
axes[2].imshow(hsv[:,:,1], cmap='gray')
axes[2].set_title("Saturation")
axes[3].imshow(hsv[:,:,2], cmap='gray')
axes[3].set_title("Value (brightness)")
for ax in axes:
    ax.axis('off')
plt.suptitle("HSV Channels — look at where the card is bright in Hue & Saturation", fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Step 2: Create a green color mask
# CNIE card green is in the HSV range roughly:
#   Hue: 30-90 (green-yellow to green)
#   Saturation: 20-255 (not too gray)
#   Value: 80-255 (not too dark)
#
# ADJUST these ranges if the mask doesn't cover the card well

lower_green = np.array([30, 20, 80])
upper_green = np.array([90, 255, 255])

mask = cv2.inRange(hsv, lower_green, upper_green)

# Clean up the mask — close small holes, remove noise
kernel = np.ones((7, 7), np.uint8)
mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)
mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(cv2.cvtColor(front, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")
axes[1].imshow(mask, cmap='gray')
axes[1].set_title("Green Mask (white = detected as card)")
# Overlay mask on original
overlay = front.copy()
overlay[mask == 0] = overlay[mask == 0] // 3  # darken non-card areas
axes[2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
axes[2].set_title("Overlay — card area highlighted")
for ax in axes:
    ax.axis('off')
plt.suptitle("Green Color Detection", fontweight='bold')
plt.tight_layout()
plt.show()

# Coverage check
total_pixels = mask.shape[0] * mask.shape[1]
card_pixels = np.count_nonzero(mask)
print(f"Card area: {card_pixels}/{total_pixels} pixels ({100*card_pixels/total_pixels:.1f}%)")
print()
print("Is the white area covering the ENTIRE card and ONLY the card?")
print("If too small: lower the Saturation/Value minimums above")
print("If too large: raise the ranges to be more strict")

In [ ]:
# Step 3: Find card contour — minAreaRect for precise corners + confidence scoring
contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contours = sorted(contours, key=cv2.contourArea, reverse=True)

if len(contours) == 0:
    print("ERROR: No contours found in green mask")
    card_contour = None
else:
    largest = contours[0]
    img_area = front.shape[0] * front.shape[1]
    contour_area = cv2.contourArea(largest)

    # === Confidence Scoring ===
    # 1. Green coverage: what % of image is green
    green_pct = 100 * np.count_nonzero(mask) / img_area

    # 2. minAreaRect gives precise rotated rectangle
    rect = cv2.minAreaRect(largest)
    (cx, cy), (w_r, h_r), angle = rect
    aspect = max(w_r, h_r) / min(w_r, h_r) if min(w_r, h_r) > 0 else 0

    # 3. Solidity: contour area vs convex hull (rectangle ≈ 1.0)
    hull = cv2.convexHull(largest)
    hull_area = cv2.contourArea(hull)
    solidity = contour_area / hull_area if hull_area > 0 else 0

    # 4. Card area as % of image
    area_pct = 100 * contour_area / img_area

    # 5. Tilt angle (normalize so 0° = perfectly aligned)
    tilt = angle
    if w_r < h_r:
        tilt = tilt + 90

    print("=== Detection Confidence ===")
    print(f"  Green coverage:  {green_pct:.1f}%  (expect 15-60%)")
    print(f"  Aspect ratio:    {aspect:.3f}  (expect ~1.585)")
    print(f"  Solidity:        {solidity:.3f}  (expect >0.85)")
    print(f"  Card area:       {area_pct:.1f}%  (expect 15-80%)")
    print(f"  Tilt angle:      {tilt:.1f}°  (expect < 5°)")

    warnings = []
    if green_pct < 10:
        warnings.append(f"Low green coverage ({green_pct:.1f}%) — card may not be fully visible")
    if abs(aspect - 1.585) > 0.3:
        warnings.append(f"Aspect ratio {aspect:.2f} far from expected 1.585")
    if solidity < 0.80:
        warnings.append(f"Low solidity {solidity:.2f} — contour not a clean rectangle")
    if area_pct < 5:
        warnings.append(f"Card too small ({area_pct:.1f}% of image)")

    if warnings:
        print(f"\n  WARNINGS:")
        for w in warnings:
            print(f"    - {w}")
    else:
        print(f"\n  All checks passed — high confidence detection")

    # === Get precise corners using minAreaRect (NOT approxPolyDP) ===
    # minAreaRect computes the mathematically optimal rotated rectangle
    # This gives much more precise corners than approxPolyDP approximation
    box = cv2.boxPoints(rect)
    card_contour = np.int0(box)

    # Draw result
    debug_img = front.copy()
    cv2.drawContours(debug_img, [card_contour], -1, (0, 0, 255), 3)
    pts = card_contour.reshape(-1, 2)
    for pt in pts:
        cv2.circle(debug_img, tuple(pt), 10, (0, 255, 0), -1)

    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(debug_img, cv2.COLOR_BGR2RGB))
    plt.title(f"minAreaRect — aspect={aspect:.3f}, solidity={solidity:.3f}, tilt={tilt:.1f}°")
    plt.axis('off')
    plt.show()

    print(f"\n4 corners: {pts.tolist()}")

In [ ]:
# Step 4: Perspective warp + crop to 856x540

def order_points(pts):
    """Order 4 points as: top-left, top-right, bottom-right, bottom-left."""
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]   # top-left = smallest sum
    rect[2] = pts[np.argmax(s)]   # bottom-right = largest sum
    d = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(d)]   # top-right = smallest difference
    rect[3] = pts[np.argmax(d)]   # bottom-left = largest difference
    return rect

NORM_W, NORM_H = 856, 540

pts = card_contour.reshape(4, 2).astype("float32")
ordered = order_points(pts)

# Destination: a perfect 856x540 rectangle
dst = np.array([
    [0, 0],
    [NORM_W - 1, 0],
    [NORM_W - 1, NORM_H - 1],
    [0, NORM_H - 1],
], dtype="float32")

# Warp
matrix = cv2.getPerspectiveTransform(ordered, dst)
cropped = cv2.warpPerspective(front, matrix, (NORM_W, NORM_H))

# Check if it needs rotation (taller than wide = wrong orientation)
ch, cw = cropped.shape[:2]
if ch > cw:
    cropped = cv2.rotate(cropped, cv2.ROTATE_90_CLOCKWISE)
    print("Auto-rotated to landscape")

# Display: before and after
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].imshow(cv2.cvtColor(front, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Original — {front.shape[1]}x{front.shape[0]}")
axes[1].imshow(cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Cropped & Normalized — {cropped.shape[1]}x{cropped.shape[0]}")
for ax in axes:
    ax.axis('off')
plt.suptitle("Card Detection Result", fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Output size: {cropped.shape[1]}w x {cropped.shape[0]}h")
print()
print("Does it look like a clean, flat, correctly oriented card?")
print("If upside down: the order_points function may need adjustment")
print("If cropped wrong: adjust the green HSV range in Step 2")

In [ ]:
# Step 5: Save the cropped card and use it for all further experiments
# This replaces the raw 'front' variable with the clean cropped version

front_raw = front.copy()  # keep original for reference
front = cropped            # all experiments below now use the clean crop

print(f"'front' is now the cropped card: {front.shape[1]}w x {front.shape[0]}h")
print("All field crop experiments (Section 6+) will use this clean image.")
print()
print("Saving to 'front_cropped.jpg' so you can use it in the calibration tool too.")
cv2.imwrite("front_cropped.jpg", cropped)
print("Saved.")

---
## Section 2 — Field Region Definitions

Field coordinates for a normalized 856x540 card image.

**Engineering approach:**
- Regions are WIDE — sized to fit ANY possible value (long names, compound names)
- PADDING adds extra margin to absorb small alignment errors from perspective warp
- Coordinates based on calibration tool data, widened for robustness
- Missing fields (address, place_of_birth_ar) need calibration on your card

In [ ]:
PADDING = 10  # pixels of extra margin around each field crop

# ============================================================
# Front Side Field Regions — WIDE to fit ANY name/value
# ============================================================
# Coordinates for the 856x540 normalized card.
# Based on calibration tool data, WIDENED for robustness:
#   - French names: x=0, w=500 (fits "EL MOUSSAOUI ALAOUI" etc.)
#   - Arabic names: x=330, w=270 (before photo at x=617)
#   - Dates/numbers: moderate width with room for variation
#   - PADDING is added automatically by test_field()
#
# If a field crops incorrectly, adjust its coordinates here
# and re-run the test cell below.
# ============================================================

FRONT_FIELDS = {
    # --- French text (left side of card) ---
    "first_name_fr":      {"x": 0,   "y": 148, "w": 500, "h": 52,  "lang": "fra", "psm": 7},
    "last_name_fr":       {"x": 0,   "y": 218, "w": 500, "h": 50,  "lang": "fra", "psm": 7},
    "date_of_birth":      {"x": 160, "y": 255, "w": 210, "h": 54,  "lang": "eng", "psm": 7},
    "place_of_birth_fr":  {"x": 0,   "y": 322, "w": 500, "h": 50,  "lang": "fra", "psm": 7},
    "expiry_date":        {"x": 200, "y": 365, "w": 200, "h": 45,  "lang": "eng", "psm": 7},

    # --- Arabic text (right side, before photo at x=617) ---
    "first_name_ar":      {"x": 330, "y": 130, "w": 270, "h": 48,  "lang": "ara", "psm": 7},
    "last_name_ar":       {"x": 330, "y": 204, "w": 270, "h": 48,  "lang": "ara", "psm": 7},

    # --- Bottom strip: card identifiers ---
    "card_number":        {"x": 590, "y": 405, "w": 220, "h": 42,  "lang": "eng", "psm": 8},
    "gender":             {"x": 800, "y": 400, "w": 56,  "h": 45,  "lang": "eng", "psm": 8},

    # --- Photo region (extraction only, no OCR) ---
    "photo":              {"x": 617, "y": 118, "w": 236, "h": 291, "lang": None,  "psm": None},
}

print(f"Defined {len(FRONT_FIELDS)} field regions (PADDING={PADDING}px)")
print(f"Target image size: 856w x 540h")
print()
for name, f in FRONT_FIELDS.items():
    if f["lang"]:
        print(f"  {name:20s}  x={f['x']:3d} y={f['y']:3d} w={f['w']:3d} h={f['h']:2d}  lang={f['lang']:7s} psm={f['psm']}")
    else:
        print(f"  {name:20s}  x={f['x']:3d} y={f['y']:3d} w={f['w']:3d} h={f['h']:3d}  (no OCR)")

---
## Section 3 — MRZ Isolation Experiment

The MRZ is the most important zone. Test it in complete isolation.

**MRZ = 3 lines x 30 chars, OCR-B font, Latin only, bottom of back side.**

In [ ]:
# Crop the MRZ strip from the back image
# The MRZ is at the bottom ~28% of the card
# ADJUST this ratio if needed after seeing the result

MRZ_RATIO = 0.72  # MRZ starts at ~72% down the card
mrz_y_start = int(back.shape[0] * MRZ_RATIO)
mrz_strip = back[mrz_y_start:, :]

print(f"Back image size: {back.shape[1]}w x {back.shape[0]}h")
print(f"MRZ crop from y={mrz_y_start} to bottom")
print(f"MRZ strip size: {mrz_strip.shape[1]}w x {mrz_strip.shape[0]}h")

plt.figure(figsize=(14, 3))
plt.imshow(cv2.cvtColor(mrz_strip, cv2.COLOR_BGR2RGB))
plt.title(f"MRZ Strip Crop (y={mrz_y_start} to bottom)")
plt.axis('off')
plt.show()

print("\nDoes the crop contain ONLY the 3 MRZ lines?")
print("If not, adjust MRZ_RATIO above and re-run this cell.")

In [ ]:
# Preprocess MRZ — binary threshold + upscale
mrz_gray = cv2.cvtColor(mrz_strip, cv2.COLOR_BGR2GRAY)
_, mrz_binary = cv2.threshold(mrz_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Try different upscale factors
fig, axes = plt.subplots(2, 2, figsize=(16, 6))

axes[0][0].imshow(cv2.cvtColor(mrz_strip, cv2.COLOR_BGR2RGB))
axes[0][0].set_title("Original")

axes[0][1].imshow(mrz_binary, cmap='gray')
axes[0][1].set_title("Otsu Binary")

mrz_2x = cv2.resize(mrz_binary, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
axes[1][0].imshow(mrz_2x, cmap='gray')
axes[1][0].set_title("Binary + 2x Upscale")

mrz_3x = cv2.resize(mrz_binary, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)
axes[1][1].imshow(mrz_3x, cmap='gray')
axes[1][1].set_title("Binary + 3x Upscale")

for row in axes:
    for ax in row:
        ax.axis('off')
plt.suptitle("MRZ Preprocessing Variants", fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# OCR the MRZ with optimal config for MRZ reading
# PSM 6 = single uniform block
# OEM 1 = LSTM only (more accurate for clean text)
# Whitelist = only MRZ-valid characters

MRZ_CONFIG = "--psm 6 --oem 1 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789<"

mrz_variants = {
    "Original":          mrz_strip,
    "Binary":            mrz_binary,
    "Binary + 2x":       mrz_2x,
    "Binary + 3x":       mrz_3x,
}

for name, img in mrz_variants.items():
    if len(img.shape) == 3:
        pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    else:
        pil = Image.fromarray(img)

    mrz_text = pytesseract.image_to_string(pil, lang="eng", config=MRZ_CONFIG)

    print(f"\n{'=' * 50}")
    print(f"MRZ — {name}")
    print(f"{'=' * 50}")
    lines = [l.strip() for l in mrz_text.strip().splitlines() if l.strip()]
    for i, line in enumerate(lines):
        ok = 'OK' if len(line) == 30 else f'BAD (expected 30)'
        print(f"  Line {i+1} ({len(line):2d} chars) [{ok}]: {line}")
    if len(lines) != 3:
        print(f"  WARNING: Got {len(lines)} lines instead of 3")

**Record:**
- Did you get 3 lines?
- Are they each 30 characters?
- Which preprocessing variant worked best?
- Are the characters recognizable as MRZ content?

---
## Section 4 — Single Field Crop Test

Test OCR on each field region defined in Section 2.
Uses the enhanced `test_field()` with padding + validation.

**PSM guide for fields:**
- PSM 7 = single text line (best for most fields)
- PSM 8 = single word (card number, gender)
- PSM 6 = text block (address, multi-line)

In [ ]:
import re

def test_field(img, x, y, w, h, field_name, lang="ara+fra", psm=7, upscale=2, padding=PADDING):
    """
    Crop a field region, preprocess, OCR, and display results.
    Applies padding around the crop to absorb alignment errors from perspective warp.
    """
    # Apply padding (clamp to image bounds)
    ih, iw = img.shape[:2]
    x1 = max(0, x - padding)
    y1 = max(0, y - padding)
    x2 = min(iw, x + w + padding)
    y2 = min(ih, y + h + padding)

    crop_img = img[y1:y2, x1:x2]

    if crop_img.size == 0:
        print(f"ERROR: Empty crop for {field_name} at ({x},{y},{w},{h})")
        print(f"Image size: {iw}w x {ih}h")
        return None

    # Preprocess: grayscale -> otsu binary -> upscale
    gray = cv2.cvtColor(crop_img, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    upscaled_img = cv2.resize(binary, None, fx=upscale, fy=upscale, interpolation=cv2.INTER_CUBIC)

    # Display
    fig, axes = plt.subplots(1, 3, figsize=(14, 2))
    axes[0].imshow(cv2.cvtColor(crop_img, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"Crop (+{padding}px pad)")
    axes[1].imshow(binary, cmap='gray')
    axes[1].set_title("Binary")
    axes[2].imshow(upscaled_img, cmap='gray')
    axes[2].set_title(f"Binary + {upscale}x")
    for ax in axes:
        ax.axis('off')
    plt.suptitle(f"{field_name}  |  lang={lang}  psm={psm}  pad={padding}", fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # OCR on all 3 versions
    config = f"--psm {psm}"
    results = {}
    for label, test_img in [("Original", crop_img), ("Binary", binary), (f"{upscale}x", upscaled_img)]:
        if len(test_img.shape) == 3:
            pil = Image.fromarray(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB))
        else:
            pil = Image.fromarray(test_img)
        text = pytesseract.image_to_string(pil, lang=lang, config=config).strip()
        results[label] = text

    # Pick best result (prefer upscaled)
    best = results.get(f"{upscale}x", "") or results.get("Binary", "") or results.get("Original", "")

    # === Validation ===
    valid = True
    validation_msg = ""

    if "date" in field_name.lower():
        if best and not re.match(r'^\d{2}[./]\d{2}[./]\d{4}$', best):
            valid = False
            validation_msg = f"Expected DD.MM.YYYY, got '{best}'"
    elif field_name == "card_number":
        if best and not re.match(r'^[A-Z]{1,3}\d{4,8}$', best):
            valid = False
            validation_msg = f"Expected letters+digits (e.g. AB12345), got '{best}'"
    elif field_name == "gender":
        if best and best.strip() not in ("M", "F", "MI"):
            valid = False
            validation_msg = f"Expected M or F, got '{best}'"
    elif "_ar" in field_name:
        has_arabic = bool(re.search(r'[\u0600-\u06FF]', best))
        if best and not has_arabic:
            valid = False
            validation_msg = f"Expected Arabic text, got '{best}'"

    # Print results
    print(f"Field: {field_name}")
    for label, text in results.items():
        marker = " << best" if label == f"{upscale}x" else ""
        print(f"  {label:12s} -> '{text}'{marker}")
    if validation_msg:
        print(f"  >> VALIDATION: {validation_msg}")
    elif best:
        print(f"  >> Validation passed")
    print()

    return {"results": results, "best": best, "valid": valid}

In [ ]:
# ============================================================
# Test ALL front side fields with the wide regions
# ============================================================
# Uses FRONT_FIELDS defined above + test_field() with padding
# Each field shows: original crop, binary, upscaled + OCR results
# ============================================================

print(f"Testing {len(FRONT_FIELDS)} fields on the cropped {front.shape[1]}x{front.shape[0]} card")
print(f"Padding: {PADDING}px on each side")
print("=" * 60)

all_results = {}

for field_name, field in FRONT_FIELDS.items():
    if field["lang"] is None:  # skip photo
        continue

    result = test_field(
        front,
        field["x"], field["y"], field["w"], field["h"],
        field_name,
        lang=field["lang"],
        psm=field["psm"],
        padding=PADDING
    )
    if result:
        all_results[field_name] = result

# === Summary Table ===
print("\n" + "=" * 60)
print("SUMMARY — All Front Side Fields")
print("=" * 60)
passed = 0
total = 0
for name, r in all_results.items():
    total += 1
    status = ">>" if r["valid"] else "!!"
    if r["valid"]:
        passed += 1
    print(f"  {status} {name:20s} -> '{r['best']}'")

print(f"\nValidation: {passed}/{total} fields passed")
if passed == total:
    print("All fields validated successfully")
else:
    print("Some fields need attention — check coordinates or OCR config")

---
## Section 5 — Back Side Fields (non-MRZ)

Test individual field crops on the back side of the card.
Back side contains: card number, personal number, father/mother names (French + Arabic).

In [ ]:
# Scale factors for back image
back_w = back.shape[1]
back_h = back.shape[0]
bsx = back_w / 856
bsy = back_h / 540

def sb(x, y, w, h):
    """Scale coordinates for back image."""
    return int(x*bsx), int(y*bsy), int(w*bsx), int(h*bsy)

print(f"Back image: {back_w}w x {back_h}h")
print(f"Scale factors: bsx={bsx:.2f}, bsy={bsy:.2f}")

In [ ]:
# Card number (repeated on back)
x, y, w, h = sb(20, 0, 400, 55)
test_field(back, x, y, w, h, "card_number_back", lang="eng", psm=8)

In [ ]:
# Father name French
x, y, w, h = sb(20, 110, 816, 40)
test_field(back, x, y, w, h, "father_name_fr", lang="fra", psm=7)

In [ ]:
# Personal number
x, y, w, h = sb(20, 55, 400, 55)
test_field(back, x, y, w, h, "personal_number", lang="eng", psm=7)

In [ ]:
# Father name Arabic
x, y, w, h = sb(20, 150, 816, 40)
test_field(back, x, y, w, h, "father_name_ar", lang="ara", psm=7)

In [ ]:
# Mother name French
x, y, w, h = sb(20, 190, 816, 40)
test_field(back, x, y, w, h, "mother_name_fr", lang="fra", psm=7)

In [ ]:
# Mother name Arabic
x, y, w, h = sb(20, 230, 816, 40)
test_field(back, x, y, w, h, "mother_name_ar", lang="ara", psm=7)

---
## Section 6 — MRZ Decoder & Validation

Decode the MRZ text into structured fields and validate check digits.
Uses ICAO 9303 standard for TD1 format (3 lines x 30 chars).

In [ ]:
def check_digit(s):
    """MRZ check digit algorithm (ICAO 9303)."""
    weights = [7, 3, 1]
    total = 0
    for i, c in enumerate(s):
        if c.isdigit():
            val = int(c)
        elif c.isalpha():
            val = ord(c.upper()) - 55  # A=10, B=11 ... Z=35
        else:  # '<'
            val = 0
        total += val * weights[i % 3]
    return total % 10


def decode_mrz(mrz_text):
    """Decode TD1 MRZ (3 lines x 30 chars) into structured fields."""
    lines = [l.strip() for l in mrz_text.strip().splitlines() if l.strip()]

    if len(lines) < 3:
        print(f"ERROR: Expected 3 lines, got {len(lines)}")
        return None

    l1, l2, l3 = lines[0][:30], lines[1][:30], lines[2][:30]

    print(f"Line 1 ({len(l1)} chars): {l1}")
    print(f"Line 2 ({len(l2)} chars): {l2}")
    print(f"Line 3 ({len(l3)} chars): {l3}")
    print()

    # Line 1: doc type, country, card number
    card_number = l1[5:14].replace("<", "")
    card_check = int(l1[14]) if l1[14].isdigit() else -1
    card_valid = check_digit(l1[5:14]) == card_check

    # Line 2: DOB, gender, expiry, nationality
    dob = l2[0:6]
    dob_check = int(l2[6]) if l2[6].isdigit() else -1
    dob_valid = check_digit(l2[0:6]) == dob_check

    gender = l2[7]

    expiry = l2[8:14]
    expiry_check = int(l2[14]) if l2[14].isdigit() else -1
    expiry_valid = check_digit(l2[8:14]) == expiry_check

    nationality = l2[15:18].replace("<", "")

    # Line 3: name
    name_parts = l3.split("<<")
    last_name = name_parts[0].replace("<", " ").strip()
    first_name = name_parts[1].replace("<", " ").strip() if len(name_parts) > 1 else ""

    # Parse dates
    def parse_date(yymmdd):
        try:
            yy, mm, dd = int(yymmdd[:2]), yymmdd[2:4], yymmdd[4:6]
            year = 1900 + yy if yy > 30 else 2000 + yy
            return f"{year}-{mm}-{dd}"
        except:
            return yymmdd

    result = {
        "doc_type":     l1[0:2].replace("<", ""),
        "country":      l1[2:5],
        "card_number":  card_number,
        "card_check":   f"{'PASS' if card_valid else 'FAIL'} (computed={check_digit(l1[5:14])}, expected={card_check})",
        "dob":          parse_date(dob),
        "dob_check":    f"{'PASS' if dob_valid else 'FAIL'} (computed={check_digit(l2[0:6])}, expected={dob_check})",
        "gender":       gender,
        "expiry":       parse_date(expiry),
        "expiry_check": f"{'PASS' if expiry_valid else 'FAIL'} (computed={check_digit(l2[8:14])}, expected={expiry_check})",
        "nationality":  nationality,
        "last_name":    last_name,
        "first_name":   first_name,
    }

    print("Decoded fields:")
    for k, v in result.items():
        print(f"  {k:15s}: {v}")

    return result

print("check_digit() and decode_mrz() defined.")
print("Run the next cell to decode the MRZ text from Section 3.")

In [ ]:
# Run the MRZ decoder on the best MRZ text from Section 3
# Make sure you ran the MRZ OCR cell first (Section 3) so mrz_text exists

if 'mrz_text' not in dir():
    print("ERROR: mrz_text not defined")
    print("Go back to Section 3 and run the MRZ OCR cell first.")
    print("The variable 'mrz_text' should contain the raw MRZ string.")
    decoded = None
else:
    print("Input MRZ text:")
    print(mrz_text)
    print()
    decoded = decode_mrz(mrz_text)

    if decoded:
        print()
        print("=" * 50)
        checks = [k for k in decoded if k.endswith("_check")]
        passed = sum(1 for k in checks if "PASS" in decoded[k])
        print(f"Check digits: {passed}/{len(checks)} passed")
        if passed == len(checks):
            print(">> All check digits valid — MRZ read correctly")
        else:
            print("!! Some check digits failed — MRZ may have OCR errors")

---
## Section 7 — Cross-Validation (Front OCR vs MRZ)

Compare what Tesseract read from the front side fields with what the MRZ decoder extracted.
If both agree, we have high confidence. If they disagree, something needs fixing.

In [ ]:
# Cross-validate front side OCR results against MRZ decoded data
# Run this AFTER running both:
#   - Section 3 (field crop test) to populate all_results
#   - Section 5 (MRZ decoder) to populate decoded

if 'decoded' not in dir() or decoded is None:
    print("ERROR: Run the MRZ decoder cell first")
elif 'all_results' not in dir() or not all_results:
    print("ERROR: Run the field crop test cell first")
else:
    print("=" * 60)
    print("CROSS-VALIDATION: Front Side OCR vs MRZ Decoded")
    print("=" * 60)

    comparisons = [
        ("first_name_fr", "first_name",  "First Name"),
        ("last_name_fr",  "last_name",   "Last Name"),
        ("date_of_birth", "dob",         "Date of Birth"),
        ("expiry_date",   "expiry",      "Expiry Date"),
        ("gender",        "gender",      "Gender"),
        ("card_number",   "card_number", "Card Number"),
    ]

    matches = 0
    total = 0

    for front_key, mrz_key, label in comparisons:
        front_val = all_results.get(front_key, {}).get("best", "").strip()
        mrz_val = str(decoded.get(mrz_key, "")).strip()

        # Normalize for comparison
        front_upper = front_val.upper()
        mrz_upper = mrz_val.upper()

        # Normalize dates: front gives DD.MM.YYYY, MRZ gives YYYY-MM-DD
        if "date" in front_key.lower():
            if re.match(r'\d{2}\.\d{2}\.\d{4}', front_val):
                parts = front_val.split(".")
                front_norm = f"{parts[2]}-{parts[1]}-{parts[0]}"
            else:
                front_norm = front_upper
            match = front_norm.upper() == mrz_upper
        else:
            match = front_upper == mrz_upper

        total += 1
        if match:
            matches += 1

        status = "MATCH" if match else "MISMATCH"
        print(f"  {label:15s}: front='{front_val}' vs mrz='{mrz_val}'  [{status}]")

    print(f"\nResult: {matches}/{total} fields match")
    if matches == total:
        print(">> All fields consistent — high confidence in OCR results")
    elif matches >= total - 1:
        print(">> Minor mismatch — likely OCR noise, check the failing field")
    else:
        print("!! Multiple mismatches — check field regions or detection quality")

---
## Section 8 — Checklist

After running all experiments, answer these:

**MRZ:**
- [ ] Did Tesseract read all 3 lines?
- [ ] Were lines exactly 30 characters?
- [ ] Did check digits pass?
- [ ] Which preprocessing worked best?

**Front side fields:**
- [ ] All field crops look correct? (check the visual outputs)
- [ ] French names readable?
- [ ] Arabic names readable?
- [ ] Dates in DD.MM.YYYY format?
- [ ] Card number reads correctly?

**Back side fields:**
- [ ] Father/mother names readable?
- [ ] Personal number readable?

**Cross-validation:**
- [ ] Front OCR matches MRZ decoded data?
- [ ] Any persistent mismatches to investigate?

**Decision:** Tesseract is good enough / need hybrid approach / need different coordinates